In [1]:
!pip install nltk


[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install textblob


[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
!python -m nltk.downloader allimport

/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py:127: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
  warn(RuntimeWarning(msg))
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /Users/admin/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /Users/admin/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /Users/admin/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /Users/admin/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/aver

In [ ]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download()

In [5]:
import nltk
import textblob
import pandas as pd
import numpy as np

### Read and overview

In [7]:
df = pd.read_csv('./input_data/spam.csv', encoding='latin-1')

In [8]:
df.head(2)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN


In [9]:
df.tail(2)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN
5571,ham,Rofl. Its true to its name,NaN,NaN,NaN


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


### Preprocessing

In [11]:
df = df[['v2', 'v1']]

In [12]:
df.duplicated().any()

True

In [13]:
df.drop_duplicates(inplace=True)

In [14]:
df.shape

(5169, 2)

In [15]:
df.v1.value_counts()

ham     4516
spam     653
Name: v1, dtype: int64

### Split into train/test

In [16]:
train_data = df.sample(frac=0.8, random_state=1)
test_data = df.drop(train_data.index)

In [17]:
train_data.shape, test_data.shape

((4135, 2), (1034, 2))

In [21]:
train_sample = train_data.values
test_sample = test_data.values

In [28]:
train_data.to_records(index=False)

rec.array([("I've been barred from all B and Q stores for life!?This twat in orange dungerees came up to me and asked if I wanted decking? So I got the first punch in!!", 'ham'),
           ('Gam gone after outstanding innings.', 'ham'),
           ('Am okay. Will soon be over. All the best', 'ham'), ...,
           ('I love ya too but try and budget your money better babe. Gary would freak on me if he knew', 'ham'),
           ('We have sent JD for Customer Service cum Accounts Executive to ur mail id, For details contact us', 'ham'),
           ('FREE GAME. Get Rayman Golf 4 FREE from the O2 Games Arcade. 1st get UR games settings. Reply POST, then save & activ8. Press 0 key for Arcade. Termsapply', 'spam')],
          dtype=[('v2', 'O'), ('v1', 'O')])

### Build model

In [22]:
from textblob.classifiers import NaiveBayesClassifier

In [24]:
model = NaiveBayesClassifier(train_sample)

### Evaluate result

In [30]:
model_score = model.accuracy(test_sample)

In [31]:
print(f'Accuracy of model: {model_score}')

Accuracy of model: 0.9835589941972921


### Run with actual data

In [39]:
ham_sent = 'I will work from home tomorrow, please send me document...'
spam_sent = 'Call 01234 for subscribe our new services with discount 50%'

In [34]:
model.classify(ham_sent)

'ham'

In [40]:
model.classify(spam_sent)

'spam'

### Show informative feature

In [36]:
model.show_informative_features()

Most Informative Features
             contains(T) = True             spam : ham    =    197.8 : 1.0
          contains(STOP) = True             spam : ham    =    188.7 : 1.0
            contains(16) = True             spam : ham    =    138.7 : 1.0
         contains(Nokia) = True             spam : ham    =    106.9 : 1.0
       contains(service) = True             spam : ham    =    102.3 : 1.0
           contains(Txt) = True             spam : ham    =     81.1 : 1.0
          contains(Free) = True             spam : ham    =     77.8 : 1.0
          contains(line) = True             spam : ham    =     66.8 : 1.0
           contains(SMS) = True             spam : ham    =     61.4 : 1.0
         contains(award) = True             spam : ham    =     61.4 : 1.0


### Wrap into Textblob

In [41]:
from textblob import TextBlob

In [43]:
TextBlob(ham_sent, classifier=model).classify()

'ham'

In [44]:
TextBlob(spam_sent, classifier=model).classify()

'spam'